# BASTION Paper Figures: NY Electricity Demand (Figures 3–5)

Reproduces Figures 3, 4, and 5 from the BASTION paper.

- **Figure 3**: Raw daily electricity demand time series.
- **Figure 4**: BASTION decomposition — Trend, Weekly seasonality (zoomed), Yearly seasonality, Volatility.
- **Figure 5**: Yearly seasonality comparison TBATS/MSTL/STR/BASTION (first 730 days).

`fit_BASTION` with `Ks=[7,365]`, `Outlier=True`, `sparse=True`, `obsSV='SV'`, `nsave=5000, nburn=5000`.
**Warning: this cell takes approximately 1–2 hours.**

In [ ]:
import os, sys, warnings
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
EXTDATA_DIR  = os.path.join(PROJECT_DIR, 'BASTION', 'inst', 'extdata')
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION
from pybastion.datasets import load_NYelectricity

print(f'Project root : {PROJECT_DIR}')

In [ ]:
_raw = load_NYelectricity()
elec = _raw.rename(columns={'Date': 'date', 'Demand': 'demand'}).copy()
elec['date'] = pd.to_datetime(elec['date'])
print(f'Electricity data : {elec.shape[0]} observations')
print(f'  Date range : {elec["date"].min().date()} to {elec["date"].max().date()}')
print(f'  Demand range : {elec["demand"].min():.1f} to {elec["demand"].max():.1f} MWh')

In [ ]:
print('Fitting BASTION (nsave=5000, nburn=5000, nchains=1) ... (~1-2 h)')
result = fit_BASTION(
    elec['demand'].values,
    Ks=[7, 365],
    Outlier=True,
    sparse=True,
    obsSV='SV',
    cl=0.95,
    nsave=5000,
    nburn=5000,
    nchains=1,
    seed=100,
)
summary = result['summary']
print('Done. Summary keys:', list(summary.keys()))

## Figure 3: Raw Electricity Demand

Full daily series from 2020-01-01 to 2022-12-31.

In [ ]:
dates  = elec['date'].values
demand = elec['demand'].values

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(dates, demand, linewidth=0.7, color='black')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Electricity Demand (MWh)', fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure3_electricity_raw.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure3_electricity_raw.pdf')

## Figure 4: BASTION Decomposition

2x2 grid: Trend (full period), Weekly seasonality (days 285-385 zoomed), Yearly seasonality (full period), Volatility (observation-level std dev).

In [ ]:
dates  = elec['date'].values
demand = elec['demand'].values

trend_mean = np.asarray(summary['Trend_sum']['Mean']).ravel()
trend_lo   = np.asarray(summary['Trend_sum']['CR_lower']).ravel()
trend_hi   = np.asarray(summary['Trend_sum']['CR_upper']).ravel()
s7_mean    = np.asarray(summary['Seasonal7_sum']['Mean']).ravel()
s7_lo      = np.asarray(summary['Seasonal7_sum']['CR_lower']).ravel()
s7_hi      = np.asarray(summary['Seasonal7_sum']['CR_upper']).ravel()
s365_mean  = np.asarray(summary['Seasonal365_sum']['Mean']).ravel()
s365_lo    = np.asarray(summary['Seasonal365_sum']['CR_lower']).ravel()
s365_hi    = np.asarray(summary['Seasonal365_sum']['CR_upper']).ravel()
volat_mean = np.asarray(summary['Volat_sum']['Mean']).ravel()
volat_lo   = np.asarray(summary['Volat_sum']['CR_lower']).ravel()
volat_hi   = np.asarray(summary['Volat_sum']['CR_upper']).ravel()

zoom   = slice(284, 385)
dates_z = dates[zoom]

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

ax = axes[0, 0]
ax.fill_between(dates, trend_lo, trend_hi, color='grey', alpha=0.4)
ax.plot(dates, trend_mean, linewidth=1.2, color='black')
ax.set_title('Trend', fontsize=12)
ax.set_ylabel('MWh', fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = axes[0, 1]
ax.fill_between(dates_z, s7_lo[zoom], s7_hi[zoom], color='grey', alpha=0.4)
ax.plot(dates_z, s7_mean[zoom], linewidth=1.2, color='black')
ax.set_title('Weekly Seasonality (days 285-385)', fontsize=12)
ax.set_ylabel('MWh', fontsize=10)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax = axes[1, 0]
ax.fill_between(dates, s365_lo, s365_hi, color='grey', alpha=0.4)
ax.plot(dates, s365_mean, linewidth=1.2, color='black')
ax.set_title('Yearly Seasonality', fontsize=12)
ax.set_ylabel('MWh', fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = axes[1, 1]
ax.fill_between(dates, volat_lo, volat_hi, color='grey', alpha=0.4)
ax.plot(dates, volat_mean, linewidth=1.2, color='black')
ax.set_title('Observation Volatility (Std Dev)', fontsize=12)
ax.set_ylabel('MWh', fontsize=10)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

for ax in axes.flatten():
    ax.tick_params(axis='x', labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure4_bastion_decomposition.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure4_bastion_decomposition.pdf')

## Figure 5: Yearly Seasonality Comparison (First 730 Days)

4-panel comparison of TBATS, MSTL, STR, and BASTION yearly seasonal components. R reference files loaded when available.

In [ ]:
idx_730   = slice(0, 730)
dates_730 = elec['date'].values[idx_730]

b_s365    = s365_mean[idx_730]
b_s365_lo = s365_lo[idx_730]
b_s365_hi = s365_hi[idx_730]

has_tbats = has_mstl = has_str = False
tbats_s365 = mstl_s365 = str_s365 = None
str_s365_lo = str_s365_hi = None

try:
    import rdata

    def _ts(obj, attrs):
        data, dim = np.array(obj), attrs.get('dim', None)
        cols = (attrs.get('dimnames') or [None, None])[1]
        if dim is not None and len(dim) == 2:
            data = data.reshape(dim, order='F')
            return pd.DataFrame(data, columns=list(cols) if cols else None)
        return pd.Series(data)

    _cd    = {**rdata.conversion.DEFAULT_CLASS_MAP, 'ts': _ts}
    _parse = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)), constructor_dict=_cd)
    _praw  = lambda f: rdata.conversion.convert(
        rdata.parser.parse_file(os.path.join(EXTDATA_DIR, f)))

    tb = _parse('TBATScomp_elec.rds')
    tbats_s365 = np.asarray(tb['season2']).ravel()[idx_730]
    has_tbats  = True

    ms = _parse('MSTL_elec.rds')
    mstl_s365 = np.asarray(ms['Seasonal365']).ravel()[idx_730]
    has_mstl  = True

    sp = _praw('STR_elec.rds')['output']['predictors']
    str_s365    = np.asarray(sp[1]['data']).ravel()[idx_730]
    str_s365_lo = np.asarray(sp[1]['lower']).ravel()[idx_730]
    str_s365_hi = np.asarray(sp[1]['upper']).ravel()[idx_730]
    has_str     = True
    print('R reference outputs loaded.')
except Exception as e:
    warnings.warn(f'R references unavailable ({e}). Panels will be blank except BASTION.')

panels = [
    ('TBATS',   tbats_s365, None,        None,        has_tbats),
    ('MSTL',    mstl_s365,  None,        None,        has_mstl),
    ('STR',     str_s365,   str_s365_lo, str_s365_hi, has_str),
    ('BASTION', b_s365,     b_s365_lo,   b_s365_hi,   True),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, (name, s, lo, hi, ok) in zip(axes.flatten(), panels):
    if ok and s is not None:
        if lo is not None:
            ax.fill_between(dates_730, lo, hi, color='grey', alpha=0.4)
        ax.plot(dates_730, s, linewidth=1.2, color='black')
    ax.set_title(name + ('' if ok else ' (unavailable)'), fontsize=12)
    ax.set_ylabel('MWh', fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'figure5_yearly_season_comparison.pdf'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figure5_yearly_season_comparison.pdf')